In [1]:
library(readxl)
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggrepel)
library(networkD3)
library(tibble)
library(ggalluvial)
library(RColorBrewer)
library(scales)
library(viridis)

Warning message:
“package ‘readxl’ was built under R version 4.3.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘networkD3’ was built under R version 4.3.3”
Warning message:
“package ‘ggalluvial’ was built under R version 4.3.3”
Warning message:
“package ‘scales’ was built under R version 4.3.3”
Warning message:
“package ‘viridis’ was built under R version 4.3.3”
Loading required package: viridisLite


Attaching package: ‘viridis’


The following object is masked from ‘package:scales’:

    viridis_pal




In [2]:
platelet <- read.table("/vol/projects/CIIM/cohorts_old/500FG/Molecular_phenotype/500FG_log2_platelet_count.txt", header = TRUE, sep = "", stringsAsFactors = FALSE)
head(platelet)
dim(platelet)
any(is.na(platelet))
sum(is.na(platelet))

,PLT_AP1,PLT_AF1,PLT_AP_AUC,PLT_AF_AUC,PLT_CP_AUC,PLT_CF_AUC,PLT_PMC_med,COAG_TAT,PLT_bTG,PLT_bTGperPLTb,PLT_PLT.count
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,2.077243,NA,5.641835,NA,NA,NA,0.4956952,8.387587,12.27862,NA,NA
HV002,2.678072,NA,6.046797,NA,NA,NA,0.5160151,5.835419,11.88810,NA,NA
HV003,2.070389,NA,5.226894,NA,NA,NA,0.5655972,11.829445,15.40348,7.724036,7.679480
HV004,2.435629,NA,5.811728,NA,NA,NA,0.7824086,12.412940,15.71530,7.633141,8.082149
HV005,2.395063,NA,5.872583,NA,NA,NA,0.5849625,11.119590,13.56643,NA,NA
HV006,2.462052,NA,5.985045,NA,NA,NA,0.4646683,12.257005,15.65375,7.509062,8.144658


[1] 534  11

[1] TRUE

[1] 804

In [3]:
basicPhenos = read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/Age_group_basicPhenos.csv")
rownames(basicPhenos) = basicPhenos$ID_500fg

## Filter a feature matrix and phenotype table to shared sample IDs and align rows in the same order
prep_filter_match <- function(feature_mat, basicPhenos) {
  idx <- rownames(feature_mat)[rownames(feature_mat) %in% rownames(basicPhenos)]  # keep feature_mat order
  feature_filter <- feature_mat[idx, , drop = FALSE]
  pheno_match <- basicPhenos[rownames(feature_filter), , drop = FALSE]
  list(feature_filter = feature_filter, basicPhenos_filter_match = pheno_match)
}
res <- prep_filter_match(platelet, basicPhenos)
platelet_filter <- res$feature_filter
basicPhenos_filter_match <- res$basicPhenos_filter_match
##one missing gender value
basicPhenos_filter_match$Gender <- trimws(basicPhenos_filter_match$Gender)
basicPhenos_filter_match$Gender[basicPhenos_filter_match$Gender == ""] <- NA

head(platelet_filter)
dim(platelet_filter)
head(basicPhenos_filter_match)
dim(basicPhenos_filter_match)

,PLT_AP1,PLT_AF1,PLT_AP_AUC,PLT_AF_AUC,PLT_CP_AUC,PLT_CF_AUC,PLT_PMC_med,COAG_TAT,PLT_bTG,PLT_bTGperPLTb,PLT_PLT.count
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,2.077243,NA,5.641835,NA,NA,NA,0.4956952,8.387587,12.27862,NA,NA
HV002,2.678072,NA,6.046797,NA,NA,NA,0.5160151,5.835419,11.88810,NA,NA
HV003,2.070389,NA,5.226894,NA,NA,NA,0.5655972,11.829445,15.40348,7.724036,7.679480
HV004,2.435629,NA,5.811728,NA,NA,NA,0.7824086,12.412940,15.71530,7.633141,8.082149
HV005,2.395063,NA,5.872583,NA,NA,NA,0.5849625,11.119590,13.56643,NA,NA
HV006,2.462052,NA,5.985045,NA,NA,NA,0.4646683,12.257005,15.65375,7.509062,8.144658


[1] 527  11

,X,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV001,304,HV001,male,27,178,70,22.09317,Married,University degree,European,Once a month,Yes,No,24,7,204,2013,Group 5
HV002,305,HV002,male,26,178,68,21.46194,Cohabitation,University degree,European,Rarely,No,Yes,24,7,204,2013,Group 4
HV003,257,HV003,female,24,173,68,22.72044,Married,University degree,European,Rarely,Yes,Yes,26,8,236,2013,Group 4
HV004,258,HV004,female,20,173,65,21.71807,Married,University degree,European,Rarely,Yes,No,26,8,236,2013,Group 2
HV005,254,HV005,female,22,178,64,20.19947,Married,University student,European,Once a month,Yes,Yes,5,9,245,2013,Group 3
HV006,530,HV006,female,24,168,58,20.54989,Married,University degree,European,Rarely,Yes,No,NA,NA,NA,NA,Group 4


[1] 527  18

In [4]:
####model 2: age + platelet using FDR age not scale + gender
all_results <- list()

for (i in seq_len(ncol(platelet_filter))) {

  # Build a minimal modeling table
  data <- data.frame(
    feature = platelet_filter[, i],
    age = basicPhenos_filter_match$Age,
    Gender = factor(basicPhenos_filter_match$Gender)  # convert chr to factor
  )

  # Linear regression
  mod <- lm(feature ~ age + Gender, data = data)

  # Extract coefficient and p-value for age (not Gender)
  cf <- summary(mod)$coefficients
  res <- cf["age", c("Estimate", "Pr(>|t|)")]

  # Store results
  result <- data.frame(
    feature  = colnames(platelet_filter)[i],
    estimate = res["Estimate"],
    p        = res["Pr(>|t|)"]
  )

  # Signed -log10(p)
  result$value <- -log10(result$p) * result$estimate

  all_results[[i]] <- result
}

final_result <- do.call(rbind, all_results)
final_result$padj <- p.adjust(final_result$p, method = "BH")
final_result$sig <- ifelse(final_result$padj < 0.05, "sig", "no")

head(final_result)
dim(final_result)
table(final_result$sig)
write.csv(final_result, file = "/vol/projects/yzhang/500FG_aging/output/09_platelet/platelet_age_gender_FDR.csv")

,feature,estimate,p,value,padj,sig
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Estimate,PLT_AP1,-0.0009190843,0.31381957,-0.0004625935,0.3835573,no
Estimate1,PLT_AF1,-0.0020392459,0.04864233,-0.0026775006,0.2012413,no
Estimate2,PLT_AP_AUC,0.0015022982,0.27211606,0.0008491678,0.3835573,no
Estimate3,PLT_AF_AUC,0.0022218860,0.26333792,0.0012875531,0.3835573,no
Estimate4,PLT_CP_AUC,-0.0011290399,0.45437515,-0.0003867926,0.4998127,no
Estimate5,PLT_CF_AUC,-0.0021594671,0.28776502,-0.0011681896,0.3835573,no


[1] 11  6


no 
11 